# Pairs trading algorithm with backtesting and metrics between Crude and Brent


__Import modules__

In [1]:
import yfinance as yf
import numpy as np
import pandas as pd
from datetime import datetime
import copy
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
import matplotlib.pyplot as plt
from sklearn import model_selection as ms
from sklearn import linear_model as lm

ModuleNotFoundError: No module named 'yfinance'

## Creating strategy (regression, signals):

Load assets from tickers and plot

In [ ]:
asset_1 = 'CL=F' #Crude Oil
asset_2 = 'BZ=F' #Brent Oil
tickers = [asset_1, asset_2]

start_date = datetime(2007, 1, 1)
end_date = datetime(2020, 1, 1)
data = yf.download(tickers, start=start_date, end=end_date)['Close'].dropna()
print(data.head())

Failed to get ticker 'XOM' reason: Expecting value: line 1 column 1 (char 0)
Failed to get ticker 'CVX' reason: Expecting value: line 1 column 1 (char 0)
[*********************100%***********************]  1 of 2 completed

2 Failed downloads:
['CVX', 'XOM']: YFTzMissingError('$%ticker%: possibly delisted; no timezone found')
[*********************100%***********************]  1 of 2 completed

Empty DataFrame
Columns: [CVX, XOM]
Index: []


Load into training and testing arrays and plot time series, print correlation and plot correlation heatmap

In [ ]:
train, test = ms.train_test_split(data, shuffle = False, test_size = 0.2)
print(data.head())

plt.figure(figsize=(14, 7))
plt.plot(train.index, train[asset_1], label='Crude Oil (CL=F)', color='blue', alpha=0.7)
plt.plot(train.index, train[asset_2], label='Brent Oil (BZ=F)', color='green', alpha=0.7)
plt.title('Training Data: Crude Oil and Brent Oil Prices')
plt.xlabel('Date')
plt.ylabel('Price')
plt.legend()
plt.grid()
plt.show()

# Calculate and print correlation
correlation = train.corr()
print("Correlation between Crude Oil and Brent Oil:")
print(correlation)

# Plot heatmap of correlation
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', cbar=True, fmt=".2f")
plt.title('Correlation Heatmap')
plt.show()

ValueError: With n_samples=0, test_size=0.2 and train_size=None, the resulting train set will be empty. Adjust any of the aforementioned parameters.

Linear regression:

In [ ]:
X = train[asset_1].values.reshape(-1, 1)  # Crude Oil prices as independent variable
Y = train[asset_2].values  # Brent Oil prices as dependent variable

model = lm.LinearRegression()
model.fit(X, Y)

# Calculate gradient, intercept, and R^2 coefficient
gradient = model.coef_[0]
intercept = model.intercept_
R2_coeff = model.score(X, Y)


print("R^2 Coefficient: ", R2_coeff)
print("Gradient: ", gradient)
print("Intercept: ", intercept)

Generate basic signal and plot:

In [ ]:
x_scores_entering = np.linspace(.1, -5, 1000)
x_scores_entering_2 = np.linspace(-.1, 5, 1000)
x_scores_exitting = np.linspace(-5, .1, 1000)
x_scores_exitting_2 = np.linspace(5, -.1, 1000)


def basic_signal(z_score_array, trigger=1, exit=0):
    signal = np.zeros(len(z_score_array))
    position = 0  # 0 = no position, 1 = long, -1 = short

    for i, element in enumerate(z_score_array):
        if position == 0:
            if element > trigger:
                signal[i] = -1  # enter short
                position = -1
            elif element < -trigger:
                signal[i] = 1   # enter long
                position = 1

        elif position == -1:  # currently short
            if element < exit:
                signal[i] = 0  # exit short
                position = 0
            else:
                signal[i] = -1  # stay short

        elif position == 1:  # currently long
            if element > -exit:
                signal[i] = 0  # exit long
                position = 0
            else:
                signal[i] = 1  # stay long

    return signal



plt.plot(x_scores_exitting, basic_signal(x_scores_exitting), label = "Exitting positions",c =  "r")
plt.plot(x_scores_exitting_2, basic_signal(x_scores_exitting_2), c = "r")
plt.plot(x_scores_entering, basic_signal(x_scores_entering), label = "Entering positions",c =  "b")
plt.plot(x_scores_entering_2, basic_signal(x_scores_entering_2), c = "b")
plt.xlabel("z score")
plt.ylabel("Trade Signal")
plt.legend()
plt.title("Basic Signal")
plt.show()


Generate tanh signal and plot:

In [ ]:
def tanh_signal(z_score_array, z_score_scaling = 2, z_score_trigger = 0.5):
  signal = np.zeros(len(z_score_array))
  entered_long = False
  entered_short = False
  for i, element in enumerate(z_score_array):
    if not entered_long and not entered_short and element > z_score_trigger:
        signal[i] = -np.tanh(element/z_score_scaling)
        entered_short = True

    elif not entered_long and not entered_short and element < -z_score_trigger:
      signal[i] = -np.tanh(element/z_score_scaling)
      entered_long = True

    elif element > -z_score_trigger and entered_long:
      signal[i] = 0
      entered_long = False

    elif element < z_score_trigger and entered_short:
      signal[i] = 0
      entered_short = False

    else:
      if i == 0:
        signal[i] = 0
      elif entered_short:
        signal[i] = min([signal[i-1], -np.tanh(element/z_score_scaling)])
      elif entered_long:
        signal[i] = max([signal[i-1], -np.tanh(element/z_score_scaling)])
      else:
        signal[i] = 0

  return signal


plt.plot(x_scores_exitting, tanh_signal(x_scores_exitting), label = "Exitting positions",c =  "r")
plt.plot(x_scores_exitting_2, tanh_signal(x_scores_exitting_2), c = "r")
plt.plot(x_scores_entering, tanh_signal(x_scores_entering), label = "Entering positions",c =  "b")
plt.plot(x_scores_entering_2, tanh_signal(x_scores_entering_2), c = "b")
plt.xlabel("z score")
plt.legend()
plt.ylabel("Trade Signal")
plt.title("Tanh Signal")
plt.show()

Last plots of signals for understanding

In [ ]:
train['Spread'] = train[asset_2] - (gradient * train[asset_1] + intercept)
train['Z-Score'] = (train['Spread'] - train['Spread'].mean()) / train['Spread'].std()

plt.plot(train.index, train['Z-Score'], label='Z-Scores', color='black', alpha=0.7)
plt.plot(train.index, basic_signal(train['Z-Score'].values), label='Basic Signal', color='blue')
plt.plot(train.index, tanh_signal(train['Z-Score'].values), label='Tanh Signal', color='red')

## Backtesting:

__backtest:__   Takes in a pair of time series, and the trade signal generated each day (without shifting - in practice we have to shift the signal by 1 so that day 1's signal affects the returns of the strategy on day1 --> day2).

It also has an optional parameter cumulative_returns which when True, shows how much money you'd make on $100 historically with this strategy, or returns the daily % PnL if __False__.

In [ ]:
def backtest(pair_time_series, unshifted_signal, cumulative_returns = True):
  asset_1 = pair_time_series.iloc[:,0]
  asset_2 = pair_time_series.iloc[:,1]

  change_in_price_asset_1 = asset_1.diff()
  change_in_price_asset_1[0] = 0
  change_in_price_asset_2 = asset_2.diff()
  change_in_price_asset_2[0] = 0

  signal_asset_1 = -unshifted_signal*gradient
  signal_asset_2 = unshifted_signal

  shifted_signal_asset_1 = np.roll(signal_asset_1,1)
  shifted_signal_asset_1[0] = 0


  shifted_signal_asset_2 = np.roll(signal_asset_2,1)
  shifted_signal_asset_2[0] = 0


  daily_returns_asset_1 = change_in_price_asset_1*shifted_signal_asset_1
  daily_returns_asset_2 = change_in_price_asset_2*shifted_signal_asset_2

  daily_returns = daily_returns_asset_1 + daily_returns_asset_2

  total_holdings = asset_1 + asset_2/gradient

  daily_percentage_returns = daily_returns/total_holdings

  if cumulative_returns:
    cumulative_pct_returns = 100*np.cumprod(1+daily_percentage_returns)
    return cumulative_pct_returns

  else:
    return daily_percentage_returns

Calculating Z-scores of spread:


In [ ]:

train_residual = train['Spread']

def create_test_z_scores(pair_time_series = test):
  asset_1 = pair_time_series.iloc[:,0]
  asset_2 = pair_time_series.iloc[:,1]

  residual = asset_2 - (gradient*asset_1+intercept)

  test_z_score = (residual - train_residual.mean())/train_residual.std()

  return test_z_score

Calculating performance metrics (Sharpe ratio):

In [ ]:
def calc_sharpe(non_cumulative_return_time_series):
  mean_daily_return = np.mean(non_cumulative_return_time_series)
  mean_yearly_return = 252*mean_daily_return
  volatility_of_returns = np.std(non_cumulative_return_time_series)
  print(mean_yearly_return, volatility_of_returns)
  return (mean_yearly_return)/(np.sqrt(252)*volatility_of_returns)

## Final testing

In [ ]:
# Calculate z-scores for train data
train['Spread'] = train[asset_2] - (gradient * train[asset_1] + intercept)
train['Z-Score'] = (train['Spread'] - train['Spread'].mean()) / train['Spread'].std()

# Generate signals for train data
basic_train_signal = basic_signal(train['Z-Score'].values)
tanh_train_signal = tanh_signal(train['Z-Score'].values)

# Backtest both strategies on train data
basic_train_returns = backtest(train, basic_train_signal, cumulative_returns=False)
tanh_train_returns = backtest(train, tanh_train_signal, cumulative_returns=False)

# Calculate Sharpe ratios
basic_sharpe_train = calc_sharpe(basic_train_returns)
tanh_sharpe_train = calc_sharpe(tanh_train_returns)

# Determine which strategy is better on train data
if basic_sharpe_train > tanh_sharpe_train:
    Better_Strategy = "Basic"
    B_S_Sharpe_Ratio = basic_sharpe_train
else:
    Better_Strategy = "Tanh"
    B_S_Sharpe_Ratio = tanh_sharpe_train

# Test data evaluation
test['Spread'] = test[asset_2] - (gradient * test[asset_1] + intercept)
test['Z-Score'] = (test['Spread'] - train['Spread'].mean()) / train['Spread'].std()

basic_test_signal = basic_signal(test['Z-Score'].values)
tanh_test_signal = tanh_signal(test['Z-Score'].values)

basic_test_returns = backtest(test, basic_test_signal, cumulative_returns=False)
tanh_test_returns = backtest(test, tanh_test_signal, cumulative_returns=False)

basic_sharpe_test = calc_sharpe(basic_test_returns)
tanh_sharpe_test = calc_sharpe(tanh_test_returns)

# Check if the better strategy on train is also better on test
if (Better_Strategy == "Basic" and basic_sharpe_test > tanh_sharpe_test) or \
   (Better_Strategy == "Tanh" and tanh_sharpe_test > basic_sharpe_test):
    True_or_False = "True"
else:
    True_or_False = "False"

print(Better_Strategy, B_S_Sharpe_Ratio)
print("Is it true for the test data: ", True_or_False)

# Plot cumulative returns for both strategies
plt.figure(figsize=(14, 7))
plt.plot(train.index, backtest(train, basic_train_signal), label='Basic Strategy', color='blue')
plt.plot(train.index, backtest(train, tanh_train_signal), label='Tanh Strategy', color='red')
plt.title('Cumulative Returns of Both Strategies (Train Data)')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns ($)')
plt.legend()
plt.grid()
plt.show()

plt.figure(figsize=(14, 7))
plt.plot(test.index, backtest(test, basic_test_signal), label='Basic Strategy', color='blue')
plt.plot(test.index, backtest(test, tanh_test_signal), label='Tanh Strategy', color='red')
plt.title('Cumulative Returns of Both Strategies (Test Data)')
plt.xlabel('Date')
plt.ylabel('Cumulative Returns ($)')
plt.legend()
plt.grid()
plt.show()